# Week 10 — The Transformer Architecture

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aofphy/SCI193611_ARTIFICIAL_INTELLIGENCE/blob/main/labs/w10_transformer.ipynb)

**Objective:** เข้าใจ self-attention และประกอบ multi-head attention จากศูนย์ด้วย NumPy.

รันได้ทันที (ต้องมี `numpy`).


## 1) Scaled Dot-Product & Multi-Head Attention


In [ ]:
import numpy as np

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-1, -2) / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    A = softmax(scores)
    return A @ V, A

def multi_head_attention(X, Wq, Wk, Wv, Wo, n_heads):
    T, d = X.shape; dh = d // n_heads
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    split = lambda t: t.reshape(T, n_heads, dh).transpose(1, 0, 2)
    out, _ = scaled_dot_product_attention(split(Q), split(K), split(V))
    out = out.transpose(1, 0, 2).reshape(T, d)
    return out @ Wo

## 2) Run & sanity checks
ตรวจ shape ของผลลัพธ์, น้ำหนัก attention รวมเป็น 1, และผลของ causal mask.


In [ ]:
np.random.seed(0)
T, d, H = 4, 8, 2
X = np.random.randn(T, d)
Wq, Wk, Wv, Wo = [np.random.randn(d, d) for _ in range(4)]

y = multi_head_attention(X, Wq, Wk, Wv, Wo, H)
print("MHA output shape:", y.shape)                       # (4, 8)

_, A = scaled_dot_product_attention(X[None], X[None], X[None])
print("attention rows sum to 1:", np.allclose(A.sum(-1), 1.0))

mask = np.tril(np.ones((T, T), bool))                     # causal mask
_, Ac = scaled_dot_product_attention(X, X, X, mask=mask)
print("position 0 attends only to itself:", np.allclose(Ac[0, 1:], 0))

## 3) TODO (ฝึกต่อ)
- เพิ่ม positional encoding (sinusoidal)
- ต่อ feed-forward + residual + LayerNorm ให้เป็น Transformer block สมบูรณ์
- เขียนใหม่ด้วย PyTorch `nn.MultiheadAttention` แล้วเทียบผลลัพธ์
